In [1]:
from script.model.pepbridge import PepBridge
from script.model.encoder import MaskedLM
from script.model.lora import build_model_with_lora
from script.dataset import PTDataSet, MPDataSet, MPTDataSet, \
    build_loader_for_long_tail, build_loader_uniform_by_peptide
from script.dataprocess import mk_aa_dict, mk_bv_dict
from script.utils import model_fn, encoder_load_state_dict, df_train_test_split

import pandas as pd 
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict

import torch
from torch.utils.data import DataLoader


In [2]:
aa_dict = mk_aa_dict()
bv_dcit = mk_bv_dict()

In [3]:
model = model_fn(aa_vocab_size=len(aa_dict),
                trbv_vocab_size=len(bv_dcit))

In [4]:
model = encoder_load_state_dict(model, peptide_pt_path='./doc/peptide_mlm.pt',
                                cdr3_pt_path='./doc/cdr3_mlm.pt', device='cpu')

In [5]:
model = build_model_with_lora(model=model, last_n=2, 
                      cfg_seq_pair=((8,16),(4,8)), 
                      dropout=0.1, freeze_base=True,
                      print_trainabel=False)

In [6]:
pt_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC_TCR/dataset/pt_train.csv', 
                    header=0, index_col=0)
pt_train, pt_val = df_train_test_split(pt_df, val_split=0.2, seed=42)
del pt_df

mp_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC/mp_train.csv', 
                    header=0, index_col=0)
mp_train, mp_val = df_train_test_split(mp_df, val_split=0.2, seed=42)
del mp_df

mpt_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC_TCR/dataset/mpt_train.csv', 
                    header=0, index_col=0)
mpt_train, mpt_val = df_train_test_split(mpt_df, val_split=0.2, seed=42)
del mpt_df

align_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC/random_pMHC_cdr3_train.csv', 
                    header=0, index_col=0,nrows=100000)
align_train, align_val = df_train_test_split(align_df, val_split=0.2, seed=42)
del align_df

In [7]:
imm_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/immuno-genicity/immunogenicity_train.csv', 
                    header=0, index_col=0)
imm_train, imm_val = df_train_test_split(imm_df, val_split=0.2, seed=42)

mp_contact_df = pd.read_csv('./data/mp_pdb_train.csv',  header=0, index_col=0)
mp_contact_train, mp_contact_val = df_train_test_split(mp_contact_df, val_split=0.2, seed=42)

pt_contact_df = pd.read_csv('./data/pt_pdb_train.csv', header=0, index_col=0)
pt_contact_train, pt_contact_val = df_train_test_split(pt_contact_df, val_split=0.2, seed=42)

In [8]:
mp_train_loader = DataLoader(MPDataSet(mp_df=mp_train, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=True, 
                        immunogenicity=False, contact=False, mask=None),
                        batch_size= 16, shuffle=True)

mp_val_loader = DataLoader(MPDataSet(mp_df=mp_val, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=True, 
                        immunogenicity=False, contact=False, mask=None),
                        batch_size= 16, shuffle=True)

imm_train_loader = DataLoader(MPDataSet(mp_df=imm_train, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=False, 
                        immunogenicity=True, contact=False, mask=None),
                        batch_size= 16, shuffle=True)

imm_val_loader = DataLoader(MPDataSet(mp_df=imm_val, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=False, 
                        immunogenicity=True, contact=False, mask=None),
                        batch_size= 16, shuffle=True)

mp_contact_train_loader = DataLoader(MPDataSet(mp_df=mp_contact_train, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=False, 
                        immunogenicity=False, contact=True, mask=None),
                        batch_size= 2, shuffle=True)

mp_contact_val_loader = DataLoader(MPDataSet(mp_df=mp_contact_val, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=False, 
                        immunogenicity=False, contact=True, mask=None),
                        batch_size= 2, shuffle=True)

In [21]:
pt_train_loader = build_loader_for_long_tail(
    dataset=PTDataSet(pt_train, pep_max_len=15, cdr3_max_len=20,
                            binding=True, contact=False, 
                            pep_mask=None, cdr3_mask=0.5),
    peptide_ids = pt_train.peptide, t=1e-3,
    batch_size=8192, max_per_group=2,
    alpha=0.5, mix=0.10, repeat_cap=64, seed=42,
    num_workers=0, pin_memory=False
)

pt_val_loader = build_loader_uniform_by_peptide(
    dataset=PTDataSet(pt_val, pep_max_len=15, 
                      cdr3_max_len=20,
                    binding=True, contact=False, 
                    pep_mask=None, cdr3_mask=None),
    peptide_ids = pt_val.peptide, 
    peptides_per_step=64, 
    samples_per_peptide=4, 
    seed=42,
    num_workers=0, pin_memory=False,
    ensure_full_batch=False
)



In [ ]:
mpt_train_loader =build_loader_for_long_tail(
    dataset=MPTDataSet(mpt_train, mhc_type='HLAI', 
               mhc_max_len=34, pep_max_len=15, cdr3_max_len=20,
               bv=True, binding=True, 
               pep_mask=None, cdr3_mask=0.5),
    peptide_ids = mpt_train.peptide,t=1e-3,
    batch_size=4096, max_per_group=2,
    alpha=0.5, mix=0.10, repeat_cap=64, seed=42,
    num_workers=0, pin_memory=False
)

mpt_val_loader = build_loader_uniform_by_peptide(
    dataset=MPTDataSet(mpt_val, mhc_type='HLAI', 
               mhc_max_len=34, pep_max_len=15, cdr3_max_len=20,
               bv=True, binding=True, 
               pep_mask=None, cdr3_mask=None),
    peptide_ids = mpt_val.peptide, 
    peptides_per_step=64, 
    samples_per_peptide=4, 
    seed=42,
    num_workers=0, pin_memory=False,
    ensure_full_batch=False
)

In [16]:
pt_contact_train_loader = DataLoader(PTDataSet(pt_contact_train, pep_max_len=15, cdr3_max_len=20,
                                binding=False, contact=True, 
                                pep_mask=None, cdr3_mask=None),
                                batch_size= 2, shuffle=True)

pt_contact_val_loader = DataLoader(PTDataSet(pt_contact_val, pep_max_len=15, cdr3_max_len=20,
                                binding=False, contact=True, 
                                pep_mask=None, cdr3_mask=None),
                                batch_size= 2, shuffle=True)

In [34]:
align_train_loader = DataLoader(MPTDataSet(align_train, mhc_type='HLAI', 
               mhc_max_len=34, pep_max_len=15, cdr3_max_len=20,
               bv=False, binding=False, 
               pep_mask=None, cdr3_mask=None),
               batch_size=8, shuffle=True)

align_val_loader = DataLoader(MPTDataSet(align_val, mhc_type='HLAI', 
               mhc_max_len=34, pep_max_len=15, cdr3_max_len=20,
               bv=False, binding=False, 
               pep_mask=None, cdr3_mask=None),
               batch_size=8, shuffle=True)

In [35]:
train_loaders =dict(
    align=align_train_loader, mp=mp_train_loader,pt=pt_train_loader,
    mp_contact=mp_contact_train_loader, pt_contact=pt_contact_train_loader,
    imm=imm_train_loader, mpt=mpt_train_loader
)
val_loaders =dict(
    align=align_val_loader, mp=mp_val_loader,pt=pt_val_loader,
    mp_contact=mp_contact_val_loader, pt_contact=pt_contact_val_loader,
    imm=imm_val_loader, mpt=mpt_val_loader
)

In [36]:
from script.train import _train_one_phase_multi, train_three_phases_multi_loaders,_infinite_iter
from torch.amp import autocast, GradScaler

In [37]:
train_three_phases_multi_loaders(  model,
    loaders=train_loaders,
    device="cuda",
    save_dir="./checkpoints_multi",
    epochs_A=5, epochs_B=10, epochs_C=10,
    steps_per_epoch=1000, 
    optimizer_ctor=lambda params: torch.optim.AdamW(params, lr=2e-4, weight_decay=0.01),
    grad_accum_steps=1,
    amp=True,
    new_optimizer_each_phase=True,
    log_interval=50,
    task_every = {"mp_contact": 20, "pt_contact": 20},   #
    val_loaders= val_loaders,
    eval_every_epochs=1,
    pep_align=True,
    use_lora=True,
    last_n=2,
    cfg_seq_pair=((8,16),(4,8)))

In [38]:
# iters = {k: _infinite_iter(dl) for k, dl in train_loaders.items()}
# task_every = {"mp_contact": 5, "pt_contact": 5}
# lambdas_C = dict(align=0.10, MP=0.50, PT=0.50, IMM=0.60, contact=0.10, MPT=0.60, lg=0.30)

# _train_one_phase_multi(model=model, device='cpu', 
#                        optimizer=torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01),
#                         scaler=GradScaler(enabled=True,device='cpu'), iters=iters, 
#                         steps_per_epoch=10, epochs=10, lambdas=lambdas_C,
#                         phase_name='C',save_path="./phase_C.pt",
#                         grad_accum_steps=1, amp=True, log_interval=1,
#                         task_every=task_every,
#                         val_loaders=None,   # 可选：提供验证
#                         eval_every_epochs=1,
#                         pep_align=True)